# Notebook 05 — MVP 1: Twitter-RoBERTa Text-Only Baseline on MMHS150K T1 (Iteration 2)

**Purpose.** Fine-tune the warm-started Twitter-RoBERTa (from NB04) on MMHS150K T1 binary-hate. **This number is the baseline that every later MVP is compared against** — naive fusion (MVP 3), gated fusion (MVP 4), and full system (MVP 5) all live or die on whether they beat MVP 1.

**Iteration 2 — single change.** Iteration 1 (logged in `Reports/Phase2_Modeling_Report.md §5–§6`) plateaued at test AUC = 0.7398, F1m = 0.6885, FPR = 0.3071 — under the Gomez 2019 target of ~0.81-0.83 AUC. Root-cause analysis flagged a **double-counted class-imbalance correction** as the most likely culprit: the loss applied Focal-CE (γ=2) AND a sklearn-`balanced` `pos_weight=3.57`. Focal CE already handles class imbalance by down-weighting easy/correct predictions; stacking `pos_weight` on top double-counts the correction and destabilises gradients. The symptom — train loss starting at random-init level and barely moving across 5 epochs — is consistent with that diagnosis.

**The only change in iteration 2:**

- **Iteration 1 loss:** `FocalBCE(γ=2, pos_weight=3.5739)` ← double-counted imbalance
- **Iteration 2 loss:** `FocalBCE(γ=2)` ← no `pos_weight`, no class weights

Every other knob is held constant — same warm-start, same LoRA rank 16 (unfrozen), same LR schedule, same 5 epochs, same batch + grad-accum, same seq_len 128, same seed 42. This is a controlled single-variable change so the iteration-2 result attributes cleanly to the loss fix.

A second, smaller methodological fix lands in cell 10: the F1-optimised threshold is now found on **recalibrated val probabilities** and then applied to recalibrated test, instead of being found on raw val and mis-applied to recalibrated test (the iter-1 inconsistency that collapsed the recalibrated confusion matrix to all-NotHate at threshold 0.5).

**Inputs.**
- LoRA adapter: `models/roberta_pretrain/` (warm-started in NB04, unfrozen here)
- Tokenizer: `models/roberta_pretrain/` (loaded **from here**, not from cardiffnlp)
- Labels: `data/processed/labels_parsed.csv` (149,819 rows, T1 derived in NB01)
- Official splits: `data/MMHS150K/splits/{train,val,test}_ids.txt` (Gomez 2019)

**Outputs.**
- `models/roberta_mvp1/` — LoRA adapter + T1 head (overwrites whatever is in this path; iteration-1 artefacts are preserved at `models/roberta_mvp1_iter1/`)
- `models/roberta_mvp1_best/` — written incrementally on each AUC improvement (overwritten each iteration)
- `outputs/nb05_confusion_matrix.png` — 2×2 test confusion matrix (overwrites iter-1)
- `outputs/nb05_training_curves.png` — train/val loss + val AUC + val F1 per epoch (overwrites iter-1)

**Acceptance criteria for declaring MVP 1 final after iteration 2** (per Phase2 report §6.6): test AUC ≥ 0.80 *and* FPR ≤ 0.25 on the balanced 50/50 split, recalibrated metrics report a non-degenerate confusion matrix, zero error cells. If unmet, a third iteration is required (likely targeting the warm-start ablation, D2 in the decisions queue).

In [8]:
import os, sys, json, random, gc, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, confusion_matrix)
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if torch.cuda.is_available():
    print(f'gpu mem: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')

HPARAMS = dict(
    max_len=128,
    batch_size=16,
    epochs=5,
    lora_rank=16,
    lora_lr=1e-4,
    head_lr=1e-3,
    warmup_ratio=0.1,
    gamma=2.0,
    grad_accum_steps=4,
    threshold=0.5,
    weight_decay=0.01,
    base_model='cardiffnlp/twitter-roberta-base-2022-154m',
    adapter_path='models/roberta_pretrain',
    prior_hate=0.2468,
    prior_nothate=0.7532,
    seed=SEED,
)
for k, v in HPARAMS.items():
    print(f'  {k}: {v}')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LABELS_CSV = ROOT / 'data' / 'processed' / 'labels_parsed.csv'
SPLITS_DIR = ROOT / 'data' / 'MMHS150K' / 'splits'
ADAPTER_DIR = ROOT / HPARAMS['adapter_path']
OUT_DIR = ROOT / 'outputs'
MODEL_DIR = ROOT / 'models' / 'roberta_mvp1'
BEST_DIR = ROOT / 'models' / 'roberta_mvp1_best'
for d in [OUT_DIR, MODEL_DIR, BEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('LABELS_CSV exists:', LABELS_CSV.exists())
print('SPLITS_DIR exists:', SPLITS_DIR.exists())
print('ADAPTER_DIR exists:', ADAPTER_DIR.exists())

device: cuda | Tesla T4
gpu mem: 14.56 GB
  max_len: 128
  batch_size: 16
  epochs: 5
  lora_rank: 16
  lora_lr: 0.0001
  head_lr: 0.001
  warmup_ratio: 0.1
  gamma: 2.0
  grad_accum_steps: 4
  threshold: 0.5
  weight_decay: 0.01
  base_model: cardiffnlp/twitter-roberta-base-2022-154m
  adapter_path: models/roberta_pretrain
  prior_hate: 0.2468
  prior_nothate: 0.7532
  seed: 42
ROOT: /teamspace/studios/this_studio/HateFusion
LABELS_CSV exists: True
SPLITS_DIR exists: True
ADAPTER_DIR exists: True


In [9]:
df = pd.read_csv(LABELS_CSV)
print('raw shape:', df.shape, '| nulls:', df.isna().sum().to_dict())

def read_ids(p):
    return set(int(line.strip()) for line in open(p) if line.strip())

splits = {
    'train': read_ids(SPLITS_DIR / 'train_ids.txt'),
    'val':   read_ids(SPLITS_DIR / 'val_ids.txt'),
    'test':  read_ids(SPLITS_DIR / 'test_ids.txt'),
}
print(f'official split sizes (from txt files): '
      f'train={len(splits["train"])} val={len(splits["val"])} test={len(splits["test"])}')

def assign_split(tid):
    for name, s in splits.items():
        if tid in s:
            return name
    return 'unassigned'

df['split'] = df['tweet_id'].map(assign_split)
df = df[df['split'].isin(['train', 'val', 'test'])].reset_index(drop=True)

print('\npost-join shape:', df.shape, '| nulls:', df.isna().sum().to_dict())
print('\nper-split T1 distribution:')
print(f'  {"split":<6} {"n":>7} {"%hate":>7} {"hate":>7} {"nothate":>8}')
for s in ['train', 'val', 'test']:
    sub = df[df['split'] == s]
    pct = sub['T1'].mean() * 100
    print(f'  {s:<6} {len(sub):>7} {pct:>6.2f}% {int(sub["T1"].sum()):>7} {(sub["T1"] == 0).sum():>8}')
print('\nExpected per Gomez 2019 design: train ~22% hate, val/test 50% balanced. Confirmed.')

raw shape: (149819, 10) | nulls: {'tweet_id': 0, 'tweet_text': 0, 'img_filename': 0, 'labels': 0, 'labels_str': 0, 'T1': 0, 'T2': 11710, 'T3': 0, 'n_annotators': 0, 't2_valid': 0}
official split sizes (from txt files): train=134823 val=5000 test=10000

post-join shape: (149819, 11) | nulls: {'tweet_id': 0, 'tweet_text': 0, 'img_filename': 0, 'labels': 0, 'labels_str': 0, 'T1': 0, 'T2': 11710, 'T3': 0, 'n_annotators': 0, 't2_valid': 0, 'split': 0}

per-split T1 distribution:
  split        n   %hate    hate  nothate
  train   134820  21.86%   29476   105344
  val       4999  50.01%    2500     2499
  test     10000  50.01%    5001     4999

Expected per Gomez 2019 design: train ~22% hate, val/test 50% balanced. Confirmed.


In [10]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print('tokenizer:', tokenizer.__class__.__name__, '| vocab:', tokenizer.vocab_size, '| loaded from:', ADAPTER_DIR)

class MMHS150KTextDataset(Dataset):
    """Pre-tokenized text-only dataset for MMHS150K T1.
    Hashtags, mentions, URLs preserved verbatim per CLAUDE.md §3."""

    def __init__(self, sub_df, tok, max_len):
        texts = sub_df['tweet_text'].astype(str).tolist()
        enc = tok(texts, truncation=True, padding='max_length',
                  max_length=max_len, return_tensors='pt')
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = torch.tensor(sub_df['T1'].to_numpy(dtype=np.float32))
        self.tweet_ids = torch.tensor(sub_df['tweet_id'].to_numpy(dtype=np.int64))

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'label': self.labels[idx],
            'tweet_id': self.tweet_ids[idx],
        }

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'split frames: train {train_df.shape} | val {val_df.shape} | test {test_df.shape}')
print(f'train_df nulls: {train_df.isna().sum().to_dict()}')

print('tokenising (one pass per split)...')
_t = time.time()
train_ds = MMHS150KTextDataset(train_df, tokenizer, HPARAMS['max_len'])
val_ds   = MMHS150KTextDataset(val_df,   tokenizer, HPARAMS['max_len'])
test_ds  = MMHS150KTextDataset(test_df,  tokenizer, HPARAMS['max_len'])
print(f'  done in {time.time()-_t:.1f}s. datasets: train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=HPARAMS['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=HPARAMS['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=HPARAMS['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)
print('sample (first 30 toks):', tokenizer.decode(train_ds[0]['input_ids'][:30]))

tokenizer: RobertaTokenizer | vocab: 50265 | loaded from: /teamspace/studios/this_studio/HateFusion/models/roberta_pretrain
split frames: train (134820, 11) | val (4999, 11) | test (10000, 11)
train_df nulls: {'tweet_id': 0, 'tweet_text': 0, 'img_filename': 0, 'labels': 0, 'labels_str': 0, 'T1': 0, 'T2': 9319, 'T3': 0, 'n_annotators': 0, 't2_valid': 0, 'split': 0}
tokenising (one pass per split)...
  done in 21.5s. datasets: train 134820 | val 4999 | test 10000
sample (first 30 toks): <s>@FriskDontMiss Nigga https://t.co/cAsaLWEpue</s><pad><pad><pad><pad><pad><pad><pad>


In [11]:
# Loss for MVP 1 iteration 2: Focal BCE γ=2 ONLY. No class weights. No pos_weight.
#
# Iteration 1 used FocalBCE(γ=2, pos_weight=3.5739), where pos_weight came from
# sklearn.utils.class_weight.compute_class_weight('balanced', ...) on the ~22%-hate train split
# (nothate=0.6399, hate=2.2869, ratio=3.5739). Diagnosed in Phase2 report §6.3 H1 as a
# double-counted imbalance correction: Focal γ=2 already down-weights easy/correct predictions,
# which is the standard mechanism for class imbalance. Multiplying by pos_weight on top makes
# the loss landscape unstable in the minority direction. The iter-1 symptom — train loss
# starting at the expected random-init level (~0.27) and barely moving across 5 epochs — is
# consistent with that diagnosis.
#
# Iteration 2 removes pos_weight entirely. Focal γ=2 alone is the loss.

print(f'train hate rate (recap): {train_df["T1"].mean()*100:.2f}%')
print('Loss config: Focal BCE, γ=2, NO class weights, NO pos_weight.')
print('  -> Focal γ=2 alone handles class imbalance by down-weighting easy/correct predictions.')

class FocalBCE(nn.Module):
    """Binary focal cross-entropy on logits — no class weighting.
    Iteration 2 simplification: removes the pos_weight that double-counted Focal's own
    imbalance handling in iteration 1."""

    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, target):
        logits = logits.view(-1)
        target = target.view(-1).float()
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
        p = torch.sigmoid(logits)
        pt = p * target + (1.0 - p) * (1.0 - target)
        focal = (1.0 - pt).pow(self.gamma) * bce
        return focal.mean()

criterion = FocalBCE(gamma=HPARAMS['gamma'])
print(f'criterion ready: FocalBCE(gamma={HPARAMS["gamma"]})  [no pos_weight, no class weights]')

train hate rate (recap): 21.86%
Loss config: Focal BCE, γ=2, NO class weights, NO pos_weight.
  -> Focal γ=2 alone handles class imbalance by down-weighting easy/correct predictions.
criterion ready: FocalBCE(gamma=2.0)  [no pos_weight, no class weights]


In [12]:
class T1Model(nn.Module):
    """Twitter-RoBERTa + warm-started LoRA (unfrozen) + fresh 1-logit T1 head."""

    def __init__(self, base_name, adapter_dir):
        super().__init__()
        base = AutoModel.from_pretrained(base_name)
        self.encoder = PeftModel.from_pretrained(base, adapter_dir, is_trainable=True)
        hidden = base.config.hidden_size
        self.head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        return self.head(cls)

model = T1Model(HPARAMS['base_model'], str(ADAPTER_DIR)).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'total params:     {total:,}')
print(f'trainable params: {trainable:,}  ({100*trainable/total:.3f}%)')
lora_train = sum(p.numel() for n, p in model.named_parameters() if 'lora_' in n and p.requires_grad)
head_train = sum(p.numel() for n, p in model.named_parameters() if n.startswith('head.') and p.requires_grad)
print(f'  lora trainable: {lora_train:,} | head trainable: {head_train:,}')
assert lora_train > 0, 'LoRA adapter is frozen — must be unfrozen for MVP 1 (is_trainable=True at load)'
print('LoRA UNFROZEN: confirmed')
model.encoder.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


total params:     125,236,225
trainable params: 590,593  (0.472%)
  lora trainable: 589,824 | head trainable: 769
LoRA UNFROZEN: confirmed
trainable params: 589,824 || all params: 125,235,456 || trainable%: 0.4710


In [13]:
lora_params = [p for n, p in model.named_parameters() if 'lora_' in n and p.requires_grad]
head_params = [p for n, p in model.named_parameters() if n.startswith('head.') and p.requires_grad]
other = [n for n, p in model.named_parameters() if p.requires_grad and 'lora_' not in n and not n.startswith('head.')]
assert not other, f'unexpected trainable params: {other[:5]}'

optimizer = torch.optim.AdamW(
    [
        {'params': lora_params, 'lr': HPARAMS['lora_lr']},
        {'params': head_params, 'lr': HPARAMS['head_lr']},
    ],
    weight_decay=HPARAMS['weight_decay'],
)

data_steps_per_epoch = len(train_loader)
opt_steps_per_epoch = data_steps_per_epoch // HPARAMS['grad_accum_steps']
total_opt_steps = opt_steps_per_epoch * HPARAMS['epochs']
warmup_steps = int(HPARAMS['warmup_ratio'] * total_opt_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_opt_steps
)
print(f'data steps/epoch: {data_steps_per_epoch}')
print(f'opt  steps/epoch: {opt_steps_per_epoch}  (grad_accum={HPARAMS["grad_accum_steps"]}, effective batch={HPARAMS["batch_size"]*HPARAMS["grad_accum_steps"]})')
print(f'total opt steps:  {total_opt_steps}  |  warmup: {warmup_steps}')

data steps/epoch: 8427
opt  steps/epoch: 2106  (grad_accum=4, effective batch=64)
total opt steps:  10530  |  warmup: 1053


In [14]:
scaler = torch.amp.GradScaler('cuda')
history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_auc = -1.0
best_epoch = -1

for epoch in range(1, HPARAMS['epochs'] + 1):
    model.train()
    t0 = time.time()
    running_loss, n_batches = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        y    = batch['label'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(ids, mask)
            loss = criterion(logits, y) / HPARAMS['grad_accum_steps']
        scaler.scale(loss).backward()
        running_loss += loss.item() * HPARAMS['grad_accum_steps']
        n_batches += 1
        if (step + 1) % HPARAMS['grad_accum_steps'] == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    train_loss = running_loss / max(n_batches, 1)

    model.eval()
    val_running, vprobs, vtrue = 0.0, [], []
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        for batch in val_loader:
            ids  = batch['input_ids'].to(device, non_blocking=True)
            mask = batch['attention_mask'].to(device, non_blocking=True)
            y    = batch['label'].to(device, non_blocking=True)
            logits = model(ids, mask)
            val_running += criterion(logits, y).item()
            vprobs.append(torch.sigmoid(logits.view(-1).float()).cpu().numpy())
            vtrue.append(y.cpu().numpy())
    val_loss = val_running / max(len(val_loader), 1)
    vprobs = np.concatenate(vprobs); vtrue = np.concatenate(vtrue)
    val_auc = roc_auc_score(vtrue, vprobs)
    val_pred = (vprobs >= HPARAMS['threshold']).astype(int)
    val_f1 = f1_score(vtrue, val_pred)
    elapsed = time.time() - t0

    print(f'[epoch {epoch}/{HPARAMS["epochs"]}] '
          f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
          f'val_auc={val_auc:.4f}  val_f1@0.5={val_f1:.4f}  ({elapsed:.1f}s)')
    history['epoch'].append(epoch); history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss); history['val_auc'].append(val_auc); history['val_f1'].append(val_f1)

    if val_auc > best_auc:
        best_auc = val_auc; best_epoch = epoch
        model.encoder.save_pretrained(BEST_DIR)
        torch.save({'head': model.head.state_dict(), 'epoch': epoch, 'val_auc': val_auc}, BEST_DIR / 'head.pt')
        print(f'  -> new best val_auc={val_auc:.4f}, saved to {BEST_DIR}')
    torch.cuda.empty_cache()

print(f'\nbest epoch: {best_epoch} | best val_auc: {best_auc:.4f}')

[epoch 1/5] train_loss=0.1237  val_loss=0.2000  val_auc=0.7424  val_f1@0.5=0.3442  (642.3s)
  -> new best val_auc=0.7424, saved to /teamspace/studios/this_studio/HateFusion/models/roberta_mvp1_best


KeyboardInterrupt: 

In [ ]:
del model
gc.collect(); torch.cuda.empty_cache()

base = AutoModel.from_pretrained(HPARAMS['base_model'])
best_encoder = PeftModel.from_pretrained(base, str(BEST_DIR), is_trainable=False)
head_ckpt = torch.load(BEST_DIR / 'head.pt', map_location='cpu', weights_only=False)
best_head = nn.Linear(base.config.hidden_size, 1)
best_head.load_state_dict(head_ckpt['head'])
print(f'reloaded best (epoch {head_ckpt["epoch"]}, val_auc={head_ckpt["val_auc"]:.4f})')

class _T1Eval(nn.Module):
    def __init__(self, enc, head):
        super().__init__()
        self.encoder = enc; self.head = head
    def forward(self, ids, mask):
        out = self.encoder(input_ids=ids, attention_mask=mask)
        return self.head(out.last_hidden_state[:, 0])

best_model = _T1Eval(best_encoder, best_head).to(device).eval()

all_probs, all_true = [], []
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    for batch in test_loader:
        ids  = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        y    = batch['label'].cpu().numpy()
        logits = best_model(ids, mask)
        probs = torch.sigmoid(logits.view(-1).float()).cpu().numpy()
        all_probs.append(probs); all_true.append(y)
all_probs = np.concatenate(all_probs); all_true = np.concatenate(all_true)
print(f'test probs: shape={all_probs.shape}, mean={all_probs.mean():.4f}, hate-rate(true)={all_true.mean():.4f}')

test_pred = (all_probs >= HPARAMS['threshold']).astype(int)
auc  = roc_auc_score(all_true, all_probs)
f1m  = f1_score(all_true, test_pred, average='macro')
precm = precision_score(all_true, test_pred, average='macro', zero_division=0)
recm  = recall_score(all_true, test_pred, average='macro', zero_division=0)
cm = confusion_matrix(all_true, test_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / max(fp + tn, 1)

print(f'\nTEST @ threshold={HPARAMS["threshold"]} (balanced 50/50 test set):')
print(f'  AUC-ROC       : {auc:.4f}')
print(f'  F1   (macro)  : {f1m:.4f}')
print(f'  Precision (M) : {precm:.4f}')
print(f'  Recall    (M) : {recm:.4f}')
print(f'  Confusion     : TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print(f'  FALSE POSITIVE RATE: {fpr:.4f}  (watch-item from NB04 not_cyberbullying recall)')

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['NotHate', 'Hate']); ax.set_yticklabels(['NotHate', 'Hate'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'NB05 Test CM (balanced 50/50)\nAUC={auc:.4f}  F1m={f1m:.4f}  FPR={fpr:.4f}')
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(OUT_DIR / 'nb05_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history['epoch'], history['train_loss'], '-o', label='train')
a1.plot(history['epoch'], history['val_loss'], '-o', label='val')
a1.set_xlabel('epoch'); a1.set_ylabel('Focal BCE loss')
a1.set_title('Loss'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(history['epoch'], history['val_auc'], '-o', color='green', label='val AUC')
a2.plot(history['epoch'], history['val_f1'], '-o', color='orange', label='val F1 @0.5')
a2.set_xlabel('epoch'); a2.set_ylabel('val metric')
a2.set_title('Validation AUC & F1'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'nb05_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Iteration-2 recalibration with FIXED threshold protocol.
#
# Iteration 1 inconsistency (Phase2 report §6.3 H6): the F1-optimised threshold was found on
# RAW val probabilities and then applied to RECALIBRATED test probabilities. Because the Bayes
# prior shift pushes most probabilities downward, threshold 0.5 (and any threshold found on
# raw val) maps to "almost all predictions are NotHate" on the recalibrated test, collapsing
# the recalibrated confusion matrix to (TN=4999, FP=0, FN=5001, TP=0).
#
# Iteration-2 fix: find the F1-optimised threshold on RECALIBRATED val probabilities, then
# apply that same threshold to recalibrated test probabilities. This keeps the threshold-search
# space and the test-application space consistent.

prior_hate = HPARAMS['prior_hate']
prior_nothate = HPARAMS['prior_nothate']

def recalibrate(p, ph=prior_hate, pn=prior_nothate):
    """Bayes shift from 50/50 balanced posterior to deployment prior P(hate)=ph.
    p_recal = (p * ph) / (p * ph + (1-p) * pn)"""
    num = p * ph
    den = num + (1 - p) * pn
    return num / np.maximum(den, 1e-12)

# Re-extract val probs from best model so we can both recalibrate and threshold-search
val_probs_list, val_true_list = [], []
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    for batch in val_loader:
        ids  = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        logits = best_model(ids, mask)
        val_probs_list.append(torch.sigmoid(logits.view(-1).float()).cpu().numpy())
        val_true_list.append(batch['label'].cpu().numpy())
val_probs_arr = np.concatenate(val_probs_list)
val_true_arr = np.concatenate(val_true_list)

test_probs_recal = recalibrate(all_probs)
val_probs_recal = recalibrate(val_probs_arr)

# Find F1-opt threshold on RECALIBRATED val (1% resolution across the achievable range).
# Recalibrated probs concentrate at lower values, so search over a wide grid.
best_thr, best_thr_f1 = 0.5, -1.0
for t in np.linspace(0.01, 0.99, 99):
    p = (val_probs_recal >= t).astype(int)
    fv = f1_score(val_true_arr, p)
    if fv > best_thr_f1:
        best_thr_f1 = fv
        best_thr = float(t)
print(f'F1-optimised threshold on RECALIBRATED VAL: {best_thr:.3f}  (val F1={best_thr_f1:.4f})')
print(f'  recalibrated val prob range: [{val_probs_recal.min():.4f}, {val_probs_recal.max():.4f}], mean={val_probs_recal.mean():.4f}')
print(f'  recalibrated test prob range: [{test_probs_recal.min():.4f}, {test_probs_recal.max():.4f}], mean={test_probs_recal.mean():.4f}')

def metrics_at(true_labels, probs, thr):
    pred = (probs >= thr).astype(int)
    auc_  = roc_auc_score(true_labels, probs)
    f1m_  = f1_score(true_labels, pred, average='macro')
    pm_   = precision_score(true_labels, pred, average='macro', zero_division=0)
    rm_   = recall_score(true_labels, pred, average='macro', zero_division=0)
    cm_   = confusion_matrix(true_labels, pred, labels=[0, 1])
    tn, fp, fn, tp = cm_.ravel()
    fpr_  = fp / max(fp + tn, 1)
    return dict(AUC=auc_, F1m=f1m_, Pm=pm_, Rm=rm_, FPR=fpr_,
                TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))

# Balanced test at threshold 0.5 (the canonical headline number for MVP 1)
m_bal_05 = metrics_at(all_true, all_probs, 0.5)
# Recalibrated test at F1-opt-on-recal-val threshold (deployment-distribution estimate)
m_rec_opt = metrics_at(all_true, test_probs_recal, best_thr)

print('\n=== SIDE-BY-SIDE METRICS ===')
print(f'  balanced   (50/50 test, threshold = 0.5)')
print(f'  recalibrated (P(hate)={prior_hate}, threshold = {best_thr:.3f}, F1-opt on recal val)')
print(f'  NOTE: AUC is invariant to monotone recalibration -> same value in both columns.\n')
print(f'  {"metric":<14}  {"balanced (50/50, thr=0.5)":<28}  {"recalibrated (~22% hate, thr=F1-opt)":<40}')
for key, name in [('AUC', 'AUC-ROC'), ('F1m', 'F1 (macro)'),
                  ('Pm', 'Precision (M)'), ('Rm', 'Recall (M)'), ('FPR', 'FPR')]:
    print(f'  {name:<14}  {m_bal_05[key]:<28.4f}  {m_rec_opt[key]:<40.4f}')
print(f'  CM balanced     TN={m_bal_05["TN"]:5} FP={m_bal_05["FP"]:5} FN={m_bal_05["FN"]:5} TP={m_bal_05["TP"]:5}')
print(f'  CM recalibrated TN={m_rec_opt["TN"]:5} FP={m_rec_opt["FP"]:5} FN={m_rec_opt["FN"]:5} TP={m_rec_opt["TP"]:5}')

# acceptance-criteria check (from Phase2 report §6.6)
auc_target = 0.80
fpr_target = 0.25
rec_cm_nondegen = (m_rec_opt['TP'] > 0 and m_rec_opt['TN'] > 0)
print('\n=== ACCEPTANCE CRITERIA (Phase2 report §6.6) ===')
print(f'  test AUC >= {auc_target}:        {m_bal_05["AUC"]:.4f}  {"PASS" if m_bal_05["AUC"] >= auc_target else "FAIL"}')
print(f'  test FPR <= {fpr_target}:        {m_bal_05["FPR"]:.4f}  {"PASS" if m_bal_05["FPR"] <= fpr_target else "FAIL"}')
print(f'  recalibrated CM non-degenerate: {rec_cm_nondegen}  {"PASS" if rec_cm_nondegen else "FAIL"}')

metrics_summary = {
    'iteration': 2,
    'change_vs_iter1': 'dropped pos_weight=3.5739 from loss; Focal γ=2 only. Also fixed recalibration threshold protocol (F1-opt now on recalibrated val).',
    'best_epoch': int(head_ckpt['epoch']),
    'best_val_auc': float(head_ckpt['val_auc']),
    'thresholds': {
        'balanced_headline': 0.5,
        'recal_f1_optimised_on_recal_val': best_thr,
    },
    'priors_for_recalibration': {'hate': prior_hate, 'nothate': prior_nothate},
    'balanced_test_thr_0.5':      {k: float(v) if not isinstance(v, int) else v for k, v in m_bal_05.items()},
    'recalibrated_test_thr_opt':  {k: float(v) if not isinstance(v, int) else v for k, v in m_rec_opt.items()},
    'acceptance_criteria': {
        'auc_target': auc_target, 'fpr_target': fpr_target,
        'auc_pass': bool(m_bal_05['AUC'] >= auc_target),
        'fpr_pass': bool(m_bal_05['FPR'] <= fpr_target),
        'recal_cm_nondegenerate': bool(rec_cm_nondegen),
    },
}
print('\nmetrics_summary keys:', list(metrics_summary.keys()))

In [ ]:
if MODEL_DIR.exists() and any(MODEL_DIR.iterdir()):
    for p in list(MODEL_DIR.iterdir()):
        if p.is_file():
            p.unlink()
        elif p.is_dir():
            shutil.rmtree(p)

for src in BEST_DIR.iterdir():
    dst = MODEL_DIR / src.name
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

with open(MODEL_DIR / 'hparams.json', 'w') as f:
    json.dump(HPARAMS, f, indent=2)
with open(MODEL_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
with open(MODEL_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print('Saved final MVP 1 model to', MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    if p.is_file():
        sz = p.stat().st_size
        unit = 'KB' if sz < 1024**2 else 'MB'
        scale = 1024 if unit == 'KB' else 1024**2
        print(f'  {p.name:>40s}  {sz/scale:>8.2f} {unit}')

print(f'\nNB05 / MVP 1 complete. Best val AUC = {head_ckpt["val_auc"]:.4f} (epoch {head_ckpt["epoch"]}).')
print('Downstream MVPs (2-5) compare against this number.')